# Data Quality Deep Dive - Skeptical QA, not EDA

Goal: actively look for reasons NOT to trust this table, rather than describe it.
Uses `pl.scan_parquet` (lazy) throughout, only `.collect()`ing when a result is needed.

Checks covered:
1. Cardinality sanity - do section_name/form/sic/tickers/name look like a clean,
   expected set per company, or is there drift/typos/unexpected variants?
2. Cross-field consistency - do report_year, reportDate, temporal_bin, docID,
   sentenceID, and row_hash actually agree with each other on a sample?
3. Random sentence sampling across random companies/years - read the actual text,
   not just its length, and judge whether it reads like real 10-K prose.
4. Degenerate-text detection - fragments, leftover HTML entities, suspiciously
   short/long sentences.
5. Exact-duplicate sentence detection - the same sentence text appearing under
   multiple sentenceIDs (would indicate double extraction).

In [1]:
import random
import polars as pl

pl.Config.set_tbl_rows(30)
pl.Config.set_fmt_str_lengths(200)

FACT_PATH = "../data_cache/stage1_facts/finrag_fact_sentences.parquet"
lf = pl.scan_parquet(FACT_PATH)

schema = lf.collect_schema()
n_rows = lf.select(pl.len()).collect().item()
print(f"Rows: {n_rows:,} | Columns: {len(schema)}")
for k, v in schema.items():
    print(f"  {k}: {v}")


Rows: 614,910 | Columns: 24
  cik: String
  cik_int: Int32
  name: String
  tickers: List(String)
  docID: String
  sentenceID: String
  section_ID: Int64
  section_name: String
  form: String
  sic: String
  sentence: String
  filingDate: String
  report_year: Int64
  reportDate: String
  temporal_bin: String
  likely_kpi: Boolean
  has_numbers: Boolean
  has_comparison: Boolean
  sample_created_at: Datetime(time_unit='us', time_zone='UTC')
  last_modified_date: Datetime(time_unit='us', time_zone='UTC')
  sample_version: String
  source_file_path: String
  load_method: String
  row_hash: String


## 1. Cardinality sanity checks

In [2]:
# Does every company have exactly ONE canonical name? (known risk: Meta/Facebook rename)
name_variants = (
    lf.group_by("cik_int")
    .agg(pl.col("name").n_unique().alias("n_names"), pl.col("name").unique().alias("names"))
    .filter(pl.col("n_names") > 1)
    .collect()
)
print(f"Companies with >1 distinct 'name' value: {name_variants.height}")
name_variants


Companies with >1 distinct 'name' value: 1


cik_int,n_names,names
i32,u32,list[str]
1326801,2,"[""Meta Platforms, Inc."", ""Facebook Inc""]"


In [3]:
# Does every company have exactly ONE sic code? (a real company shouldn't drift SIC over time)
sic_variants = (
    lf.group_by("cik_int")
    .agg(pl.col("sic").n_unique().alias("n_sics"), pl.col("sic").unique().alias("sics"))
    .filter(pl.col("n_sics") > 1)
    .collect()
)
print(f"Companies with >1 distinct 'sic' value: {sic_variants.height}")
sic_variants


Companies with >1 distinct 'sic' value: 1


cik_int,n_sics,sics
i32,u32,list[str]
813762,2,"[""3714"", ""2911""]"


In [4]:
# form should be exactly {"10-K"} - anything else means an amendment or wrong form slipped through
forms = lf.select("form").unique().collect()
print("distinct form values:", forms["form"].to_list())

# section_name should be exactly the 20 known ITEM_ tokens - flag anything unexpected
sections = lf.select("section_name").unique().sort("section_name").collect()
print(f"\ndistinct section_name values ({sections.height}):")
print(sections["section_name"].to_list())


distinct form values: ['10-K']

distinct section_name values (20):
['ITEM_1', 'ITEM_10', 'ITEM_11', 'ITEM_12', 'ITEM_13', 'ITEM_14', 'ITEM_15', 'ITEM_1A', 'ITEM_1B', 'ITEM_2', 'ITEM_3', 'ITEM_4', 'ITEM_5', 'ITEM_6', 'ITEM_7', 'ITEM_7A', 'ITEM_8', 'ITEM_9', 'ITEM_9A', 'ITEM_9B']


In [5]:
# tickers - any null/empty, or any company with a suspiciously large ticker list?
ticker_check = (
    lf.select(["cik_int", "name", "tickers"])
    .unique()
    .with_columns(pl.col("tickers").list.len().alias("n_tickers"))
    .collect()
)
print("rows with null/empty tickers:", ticker_check.filter(pl.col("n_tickers") == 0).height)
print("\ncompanies with >1 ticker (dual-class shares - expected for a few, e.g. Alphabet):")
print(ticker_check.filter(pl.col("n_tickers") > 1))


rows with null/empty tickers: 0

companies with >1 ticker (dual-class shares - expected for a few, e.g. Alphabet):
shape: (3, 4)
┌─────────┬───────────────────┬──────────────────────────────┬───────────┐
│ cik_int ┆ name              ┆ tickers                      ┆ n_tickers │
│ ---     ┆ ---               ┆ ---                          ┆ ---       │
│ i32     ┆ str               ┆ list[str]                    ┆ u32       │
╞═════════╪═══════════════════╪══════════════════════════════╪═══════════╡
│ 1283699 ┆ T-Mobile US, Inc. ┆ ["TMUS", "TMUSI", … "TMUSZ"] ┆ 4         │
│ 1341439 ┆ ORACLE CORP       ┆ ["ORCL", "ORCL-PD"]          ┆ 2         │
│ 1652044 ┆ Alphabet Inc.     ┆ ["GOOGL", "GOOG", … "GOOGN"] ┆ 4         │
└─────────┴───────────────────┴──────────────────────────────┴───────────┘


## 2. Cross-field consistency checks

In [6]:
# report_year should always equal reportDate's year (fiscal period end year) - not filingDate's
mismatch = (
    lf.with_columns(pl.col("reportDate").str.slice(0, 4).cast(pl.Int64).alias("reportDate_year"))
    .filter(pl.col("report_year") != pl.col("reportDate_year"))
    .select(["cik_int", "docID", "report_year", "reportDate", "reportDate_year"])
    .collect()
)
print(f"Rows where report_year != reportDate's year: {mismatch.height:,}")
if mismatch.height:
    print(mismatch.unique(subset=["docID"]))


Rows where report_year != reportDate's year: 0


In [7]:
# temporal_bin should exactly match a recomputed bin from report_year
def expected_bin(y):
    if 2006 <= y <= 2009: return "bin_2006_2009"
    if 2010 <= y <= 2015: return "bin_2010_2015"
    if 2016 <= y <= 2020: return "bin_2016_2020"
    if 2021 <= y <= 2025: return "bin_2021_2025"
    return "bin_unknown"

bin_check = (
    lf.select(["report_year", "temporal_bin"]).unique().collect()
    .with_columns(pl.col("report_year").map_elements(expected_bin, return_dtype=pl.String).alias("expected_bin"))
)
mismatches = bin_check.filter(pl.col("temporal_bin") != pl.col("expected_bin"))
print(f"Distinct (report_year, temporal_bin) combos: {bin_check.height}")
print(f"Mismatches against recomputed bin: {mismatches.height}")
mismatches


Distinct (report_year, temporal_bin) combos: 20
Mismatches against recomputed bin: 0


report_year,temporal_bin,expected_bin
i64,str,str


In [8]:
# sentenceID's embedded docID prefix should match the row's own docID - and its
# embedded item token should match section_name (minus the ITEM_ prefix) for the
# NEW pipeline's convention. Older batches may use a different convention, so this
# is checked per load_method, not assumed globally.
check_df = lf.select(["load_method", "docID", "sentenceID", "section_name"]).collect()

check_df = check_df.with_columns(
    pl.col("sentenceID").str.starts_with(pl.col("docID")).alias("docID_prefix_ok")
)
bad_prefix = check_df.filter(~pl.col("docID_prefix_ok"))
print(f"Rows where sentenceID doesn't start with its own docID: {bad_prefix.height:,}")

by_method = check_df.group_by("load_method").agg(
    pl.col("docID_prefix_ok").sum().alias("n_ok"),
    pl.len().alias("n_total"),
)
by_method


Rows where sentenceID doesn't start with its own docID: 0


load_method,n_ok,n_total
str,u32,u32
"""extract_and_convert""",188085,188085
"""edgartools_incremental""",170407,170407
"""stratified_sampling""",256418,256418


In [9]:
# row_hash recompute check - sample-based (full-table MD5 recompute over 600K+ rows
# is unnecessary for a spot-check; a random sample across companies/years is enough
# to catch a systematic problem without a slow full audit).
import hashlib

sample = lf.select(["sentenceID", "sentence", "row_hash"]).collect().sample(n=5000, seed=42)
recomputed = sample.with_columns(
    (pl.col("sentenceID") + pl.col("sentence"))
    .map_elements(lambda x: hashlib.md5(x.encode()).hexdigest(), return_dtype=pl.String)
    .alias("recomputed_hash")
)
n_mismatch = (recomputed["recomputed_hash"] != recomputed["row_hash"]).sum()
print(f"row_hash mismatches in 5,000-row random sample: {n_mismatch}")


row_hash mismatches in 5,000-row random sample: 0


## 3. Random sentence sampling - read the actual text across random companies/years

In [10]:
random.seed(7)

all_keys = lf.select(["cik_int", "name", "report_year"]).unique().collect()
sample_keys = all_keys.sample(n=18, seed=7)

rows = []
for cik, name, year in sample_keys.select(["cik_int", "name", "report_year"]).iter_rows():
    candidate = (
        lf.filter((pl.col("cik_int") == cik) & (pl.col("report_year") == year))
        .select(["cik_int", "name", "report_year", "section_name", "sentence"])
        .collect()
    )
    if candidate.height:
        rows.append(candidate.sample(n=1, seed=random.randint(0, 10_000)).row(0, named=True))

for r in rows:
    print(f"[{r['name']} | FY{r['report_year']} | {r['section_name']}]")
    print(f"  {r['sentence']}")
    print()


[Tesla, Inc. | FY2022 | ITEM_1A]
  Such regulations continue to rapidly change, which increases the likelihood of a patchwork of complex or conflicting regulations, or may delay, restrict or prohibit the availability of certain functionalities and vehicle designs, which could adversely affect our business.

[AMAZON COM INC | FY2016 | ITEM_9A]
  Any control system, no matter how well designed and operated, is based upon certain assumptions and can provide only reasonable, not absolute, assurance that its objectives will be met.

[Facebook Inc | FY2020 | ITEM_8]
  Cost of revenue also includes costs associated with partner arrangements, including traffic acquisition

[VISA INC. | FY2012 | ITEM_8]
  Indemnification obligations.

[RADIAN GROUP INC | FY2016 | ITEM_8]
  If the market value per share of our common stock, as measured under the terms of the capped call transactions, exceeds the applicable cap price of the capped call transactions, the number of shares of our common stock and/or

**Manual read of the sample above**: each sentence should read as coherent, real
10-K prose - a complete clause about the business, financials, or risk factors - not
a truncated fragment, a bare number, or leftover markup. Re-run the cell above (it
reseeds) to spot-check a different random draw if anything here looks off.

## 4. Degenerate-text detection

In [11]:
# Very short "sentences" that likely survived cleaning as noise, not real content
degenerate = (
    lf.with_columns(pl.col("sentence").str.len_chars().alias("char_len"))
    .filter(pl.col("char_len") < 15)
    .select(["cik_int", "name", "report_year", "section_name", "sentence", "char_len"])
    .collect()
)
print(f"Sentences under 15 characters: {degenerate.height:,} ({degenerate.height / n_rows:.3%} of all rows)")
degenerate.sample(n=min(15, degenerate.height), seed=1) if degenerate.height else print("none")


Sentences under 15 characters: 2,840 (0.462% of all rows)


cik_int,name,report_year,section_name,sentence,char_len
i32,str,i64,str,str,u32
1276520,"""GENWORTH FINANCIAL INC""",2023,"""ITEM_7""","""block in 2022.""",14
814585,"""MBIA INC""",2023,"""ITEM_9""","""None.""",5
890926,"""RADIAN GROUP INC""",2015,"""ITEM_1""","""Claim Denials.""",14
814585,"""MBIA INC""",2023,"""ITEM_8""","""No.""",3
890926,"""RADIAN GROUP INC""",2018,"""ITEM_1A""","""See “Item 3.""",12
890926,"""RADIAN GROUP INC""",2022,"""ITEM_1A""","""See ""Item 3.""",12
1326801,"""Meta Platforms, Inc.""",2024,"""ITEM_8""","""Ret.""",4
1403161,"""VISA INC.""",2021,"""ITEM_8""","""Marketing.""",10
34088,"""EXXON MOBIL CORP""",2015,"""ITEM_15""","""Risk Factors.""",13


In [12]:
# Leftover HTML entities / tags that should have been stripped during cleaning
import re

entity_pattern = r'&(nbsp|amp|lt|gt|quot|#\d+);|<[a-zA-Z/][^>]{0,30}>'
leftover = (
    lf.filter(pl.col("sentence").str.contains(entity_pattern))
    .select(["cik_int", "name", "report_year", "section_name", "sentence"])
    .collect()
)
print(f"Sentences with a likely leftover HTML entity/tag: {leftover.height:,}")
leftover.sample(n=min(10, leftover.height), seed=1) if leftover.height else print("none found - cleaning looks solid")


Sentences with a likely leftover HTML entity/tag: 0
none found - cleaning looks solid


## 5. Exact-duplicate sentence detection

In [13]:
# The same sentence TEXT appearing under different sentenceIDs within the same
# company+year is a real red flag for double-extraction (e.g. a section counted twice).
dupe_text = (
    lf.group_by(["cik_int", "report_year", "sentence"])
    .agg(pl.len().alias("n_occurrences"), pl.col("sentenceID").alias("sentenceIDs"))
    .filter(pl.col("n_occurrences") > 1)
    .collect()
)
print(f"Distinct (company, year, sentence) triples appearing more than once: {dupe_text.height:,}")
dupe_text.sort("n_occurrences", descending=True).head(15)


Distinct (company, year, sentence) triples appearing more than once: 23,966


cik_int,report_year,sentence,n_occurrences,sentenceIDs
i32,i64,str,u32,list[str]
1276520,2023,"""GENWORTH FINANCIAL, INC.""",118,"[""0001276520_10-K_2023_section_8_1017"", ""0001276520_10-K_2023_section_8_1043"", … ""0001276520_10-K_2023_section_8_997""]"
1276520,2020,"""GENWORTH FINANCIAL, INC.""",116,"[""0001276520_10-K_2020_section_8_1003"", ""0001276520_10-K_2020_section_8_1030"", … ""0001276520_10-K_2020_section_8_994""]"
1276520,2021,"""GENWORTH FINANCIAL, INC.""",115,"[""0001276520_10-K_2021_section_8_1002"", ""0001276520_10-K_2021_section_8_1030"", … ""0001276520_10-K_2021_section_8_982""]"
1276520,2024,"""GENWORTH FINANCIAL, INC.""",102,"[""0001276520_10-K_2024_section_8_1008"", ""0001276520_10-K_2024_section_8_1032"", … ""0001276520_10-K_2024_section_8_994""]"
1276520,2022,"""GENWORTH FINANCIAL, INC.""",100,"[""0001276520_10-K_2022_section_8_1005"", ""0001276520_10-K_2022_section_8_1022"", … ""0001276520_10-K_2022_section_8_985""]"
1273813,2021,"""Assured Guaranty Ltd.""",94,"[""0001273813_10-K_2021_section_8_1004"", ""0001273813_10-K_2021_section_8_1035"", … ""0001273813_10-K_2021_section_8_988""]"
1273813,2022,"""Assured Guaranty Ltd.""",88,"[""0001273813_10-K_2022_section_8_1001"", ""0001273813_10-K_2022_section_8_1025"", … ""0001273813_10-K_2022_section_8_968""]"
1273813,2023,"""Assured Guaranty Ltd.""",84,"[""0001273813_10-K_2023_section_8_1021"", ""0001273813_10-K_2023_section_8_1051"", … ""0001273813_10-K_2023_section_8_994""]"
1273813,2024,"""Assured Guaranty Ltd.""",81,"[""0001273813_10-K_2024_section_8_1021"", ""0001273813_10-K_2024_section_8_103"", … ""0001273813_10-K_2024_section_8_989""]"


In [14]:
# Where do these duplicates concentrate? (section, load_method breakdown)
dupe_full = (
    lf.group_by(["cik_int", "report_year", "sentence", "section_name", "load_method"])
    .agg(pl.len().alias("n"))
    .filter(pl.col("n") > 1)
    .collect()
)
excess_rows = (dupe_full["n"] - 1).sum()
print(f"Total duplicate groups: {dupe_full.height:,} | Total excess (noise) rows: {excess_rows:,} "
      f"({excess_rows / n_rows:.2%} of the table)")

print("\nConcentration by section (ITEM_7/ITEM_8 = financial statements/MD&A, prone to")
print("repeated page/table headers across pages):")
print(dupe_full.group_by("section_name").agg(
    pl.col("n").sum().alias("total_dupe_rows"), pl.len().alias("n_groups")
).sort("total_dupe_rows", descending=True))

print("\nBy load_method (is this a new-pipeline problem or a pre-existing one?):")
print(dupe_full.group_by("load_method").agg(
    pl.col("n").sum().alias("total_dupe_rows"), pl.len().alias("n_groups")
).sort("total_dupe_rows", descending=True))


Total duplicate groups: 6,608 | Total excess (noise) rows: 9,875 (1.61% of the table)

Concentration by section (ITEM_7/ITEM_8 = financial statements/MD&A, prone to
repeated page/table headers across pages):
shape: (20, 3)
┌──────────────┬─────────────────┬──────────┐
│ section_name ┆ total_dupe_rows ┆ n_groups │
│ ---          ┆ ---             ┆ ---      │
│ str          ┆ u32             ┆ u32      │
╞══════════════╪═════════════════╪══════════╡
│ ITEM_8       ┆ 7503            ┆ 2807     │
│ ITEM_7       ┆ 4967            ┆ 2024     │
│ ITEM_15      ┆ 1463            ┆ 703      │
│ ITEM_1       ┆ 865             ┆ 316      │
│ ITEM_1A      ┆ 728             ┆ 312      │
│ ITEM_4       ┆ 217             ┆ 103      │
│ ITEM_3       ┆ 119             ┆ 51       │
│ ITEM_5       ┆ 111             ┆ 55       │
│ ITEM_9A      ┆ 111             ┆ 55       │
│ ITEM_2       ┆ 71              ┆ 35       │
│ ITEM_13      ┆ 64              ┆ 29       │
│ ITEM_7A      ┆ 57              ┆ 27    

**Finding**: ~1.6% of all rows (9,875 / 614,910) are exact-duplicate sentences within
the same company+year - overwhelmingly concentrated in ITEM_7/ITEM_8 (financial
statements and MD&A), and overwhelmingly company-name-only fragments (e.g.
"GENWORTH FINANCIAL, INC." repeated 118 times in one filing year) - almost certainly
repeated page/table headers picked up as if they were real sentences, not a
sentence-splitting bug specific to any one load_method. Present across all three
load_methods (old and new pipelines alike), though `edgartools_incremental` has a
proportionally lower rate (~2.0%) than the two older batches (~2.6-3.4%). Not fixed
here - flagged as a concrete, scoped cleaning improvement for a future pass
(a company-name/page-header stripping step, likely in Stage 3's cleaning regex).

## Summary

In [15]:
print("Run all cells above and read their printed counts/samples directly - ")
print("this cell is just a placeholder reminder to eyeball the duplicate-sentence")
print("and degenerate-text results above before trusting the table for downstream use.")


Run all cells above and read their printed counts/samples directly - 
this cell is just a placeholder reminder to eyeball the duplicate-sentence
and degenerate-text results above before trusting the table for downstream use.
